# Track 3 — RAG Treasure Hunt: Reference Solution

This notebook is a **working reference baseline** for [Track 3](https://d4-arena-api-77646749251.us-central1.run.app/competitions/rag_treasure_hunt). It's intentionally simple — just enough to demonstrate every dimension of the rubric. Fork it and improve.

**The pipeline:**

1. **Load** the public chunked corpus (`index.json`) and the public questions (`questions.json`).
2. **Retrieve** top-k relevant chunks for each question with a hand-rolled BM25 scorer (no extra deps).
3. **Generate** a grounded answer with Claude Haiku, asking it to **cite** the chunk IDs it used.
4. **Parse** the cited chunk IDs out of the model's output → submission `citations`.
5. **Submit** the envelope and read your score.

**What this baseline does NOT do well:**
- Pure BM25 — no semantic / dense retrieval.
- Top-3 fixed — doesn't dynamically adapt to question difficulty.
- Best-effort citation parsing — relies on the model emitting `[chunk_id]` brackets.
- No re-ranking, no query rewriting, no caching.

Each of those is a place a thoughtful student can pull a few rubric points.

## 0. Setup

In [ ]:
%pip install -q httpx anthropic

import httpx, json, datetime, uuid, os, re, math
from collections import Counter
from anthropic import Anthropic

# ---- Configure ----
ARENA_URL  = 'https://d4-arena-api-77646749251.us-central1.run.app'
STUDENT_ID = 'asukul'  # replace with your Canvas/NetID
MODEL      = 'claude-haiku-4-5'
TOP_K      = 3

# Anthropic key — set via env var on your machine, or paste here for a test run.
client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-...'))

## 1. Load the public corpus + questions

Both files are version-controlled at:
- [`corpora/isu_course_catalog/index.json`](https://github.com/asukul/AI-arena/blob/main/corpora/isu_course_catalog/index.json)
- [`corpora/isu_course_catalog/questions.json`](https://github.com/asukul/AI-arena/blob/main/corpora/isu_course_catalog/questions.json)

We pull them via raw GitHub so the notebook is self-contained.

In [ ]:
RAW_BASE = 'https://raw.githubusercontent.com/asukul/AI-arena/main/corpora/isu_course_catalog'
index = httpx.get(f'{RAW_BASE}/index.json').json()
questions = httpx.get(f'{RAW_BASE}/questions.json').json()['questions']
chunks = index['chunks']  # {chunk_id: text}

print(f'Loaded {len(chunks)} chunks and {len(questions)} questions.')
for q in questions[:2]:
    print(f"  {q['question_id']}: {q['question']}")

## 2. Hand-rolled BM25 retriever

BM25 is the standard sparse-retrieval baseline. We're doing it by hand (no `rank_bm25` dependency) because:
1. Reading 30 lines of BM25 is more pedagogically useful than a one-line library import.
2. Once you understand the formula you can mutate it (different `k1`, `b`, IDF smoothing, etc.) without fighting library defaults.

BM25 scores how well a document matches a query, with two free knobs:
- `k1` (term-frequency saturation): how much repeated occurrences of a term in a doc help. Higher = more reward.
- `b` (length normalization): how much we penalize long documents. 0 = no penalty, 1 = full penalty.

Defaults below are the standard `(k1=1.5, b=0.75)`.

In [ ]:
_TOKEN = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str) -> list[str]:
    """Lowercase + alphanumeric-only. Cheap and good enough for course-catalog text.
    A sharper student would lemmatize and drop stopwords."""
    return [t.lower() for t in _TOKEN.findall(text)]


class BM25:
    """Standard BM25 scorer over a fixed corpus of (id, text) docs."""
    def __init__(self, docs: dict[str, str], *, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.doc_ids = list(docs.keys())
        self.tokens  = {d: tokenize(t) for d, t in docs.items()}
        self.len     = {d: len(toks) for d, toks in self.tokens.items()}
        self.avgdl   = sum(self.len.values()) / max(1, len(self.len))
        self.tf      = {d: Counter(toks) for d, toks in self.tokens.items()}
        # Document frequency: in how many docs does each term appear?
        df: Counter = Counter()
        for toks in self.tokens.values():
            df.update(set(toks))
        n = len(docs)
        # IDF with the BM25+ smoothing trick (always positive).
        self.idf = {term: math.log(1 + (n - cnt + 0.5) / (cnt + 0.5)) for term, cnt in df.items()}

    def score(self, query: str, doc_id: str) -> float:
        q_terms = tokenize(query)
        tf, dl = self.tf[doc_id], self.len[doc_id]
        s = 0.0
        for t in q_terms:
            if t not in self.idf:
                continue
            f = tf.get(t, 0)
            if f == 0:
                continue
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += self.idf[t] * num / den
        return s

    def top_k(self, query: str, k: int = 3) -> list[tuple[str, float]]:
        """Return [(doc_id, score)] for the top-k matching docs."""
        scored = [(d, self.score(query, d)) for d in self.doc_ids]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


bm25 = BM25(chunks)
for q in questions[:2]:
    print(f"\nQ {q['question_id']}: {q['question']}")
    for chunk_id, score in bm25.top_k(q['question'], k=TOP_K):
        snippet = chunks[chunk_id][:80]
        print(f"  [{score:.3f}] {chunk_id}: {snippet}…")

## 3. Generation: ask Claude Haiku for a grounded answer + citations

The system prompt does the heavy lifting:
- **"Answer using ONLY the provided chunks"** — pushes faithfulness up.
- **"Cite each chunk you use as `[chunk_id]`"** — gives us a parseable citation format.
- **"If the chunks don't contain the answer, say so explicitly"** — protects safety + faithfulness.

We pass the retrieved chunks in a tagged block so the model knows where they came from.

In [ ]:
SYSTEM_PROMPT = """You answer questions about Iowa State University's CS / DS / AI courses, using ONLY the chunks provided.

Rules:
1. Read the <chunks> block.
2. Answer the user's question in 1-3 sentences.
3. Cite EVERY chunk you used inline as [chunk_id]. Example: "CS 2010 has no prerequisites [cs2010_c2]."
4. If the chunks don't contain the answer, say "The provided chunks don't contain that information."
5. Do not invent course numbers, prerequisites, or policies that aren't in the chunks."""

def render_chunks(retrieved: list[tuple[str, float]]) -> str:
    """Render top-k retrieved chunks as a tagged block for the model."""
    lines = ['<chunks>']
    for chunk_id, _score in retrieved:
        lines.append(f'<chunk id="{chunk_id}">{chunks[chunk_id]}</chunk>')
    lines.append('</chunks>')
    return '\n'.join(lines)


_CITE = re.compile(r'\[([a-z0-9_]+)\]')

def parse_citations(text: str, valid_ids: set[str]) -> list[str]:
    """Pull bracketed IDs out of the model's output. Drop anything that
    isn't a real chunk_id — protects against the model inventing IDs."""
    seen: list[str] = []
    for cid in _CITE.findall(text):
        if cid in valid_ids and cid not in seen:
            seen.append(cid)
    return seen


def answer_question(question: str, retrieved: list[tuple[str, float]]) -> dict:
    """Returns a single Track-3 answer dict with all the rubric-graded fields."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        temperature=0,  # match the platform's judge for reproducibility
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': f'{render_chunks(retrieved)}\n\nQuestion: {question}'}],
    )
    text = response.content[0].text
    citations = parse_citations(text, set(chunks))
    retrieved_ids = [cid for cid, _ in retrieved]
    # Anthropic doesn't itemize $/request, so estimate from token counts.
    # Haiku 4.5 pricing as of 2026-05: $1/MTok input, $5/MTok output.
    usage = response.usage
    cost = (usage.input_tokens / 1e6) * 1.0 + (usage.output_tokens / 1e6) * 5.0
    return {
        'answer':              text,
        'citations':           citations,
        'retrieved_contexts':  retrieved_ids,
        'estimated_cost_usd':  round(cost, 6),
        'latency_seconds':     0.0,  # student exercise: time it with time.monotonic()
    }

## 4. Run the pipeline over every question

In [ ]:
answers = []
for q in questions:
    retrieved = bm25.top_k(q['question'], k=TOP_K)
    a = answer_question(q['question'], retrieved)
    a['question_id'] = q['question_id']
    answers.append(a)
    print(f"\n{q['question_id']} (cost ${a['estimated_cost_usd']:.4f}):")
    print(f"  retrieved: {a['retrieved_contexts']}")
    print(f"  cited:     {a['citations']}")
    print(f"  answer:    {a['answer'][:140]}…")

total_cost = sum(a['estimated_cost_usd'] for a in answers)
print(f"\nTotal estimated cost: ${total_cost:.4f}")

## 5. Submit the envelope

In [ ]:
envelope = {
    'submission_id':   f'sub_{STUDENT_ID}_{uuid.uuid4().hex[:8]}',
    'student_id':      STUDENT_ID,
    'track_id':        'rag_treasure_hunt',
    'submission_timestamp': datetime.datetime.now(datetime.UTC).isoformat(),
    'model_used':      MODEL,
    'prompt_version':  'reference-baseline-v1',
    'self_reported_strategy': f'BM25 top-{TOP_K} + grounded prompt + bracket citation parsing.',
    'track_payload':   {
        'answers': [
            {
                'question_id':        a['question_id'],
                'answer':             a['answer'],
                'citations':          a['citations'],
                'retrieved_contexts': a['retrieved_contexts'],
                'estimated_cost_usd': a['estimated_cost_usd'],
                'latency_seconds':    a['latency_seconds'],
            }
            for a in answers
        ],
    },
}

with httpx.Client(timeout=60.0) as http:
    r = http.post(f'{ARENA_URL}/submit', json=envelope)
if r.status_code == 200:
    body = r.json()
    print(f"Submitted! Used {body['submissions_today']}/{body['daily_limit']} of today's submissions.")
else:
    print(r.status_code, r.text)

In [ ]:
leaderboard = httpx.get(f'{ARENA_URL}/leaderboard/rag_treasure_hunt').json()
print('Top 10 on rag_treasure_hunt:')
for rank, entry in enumerate(leaderboard['entries'][:10], start=1):
    marker = ' <-- you' if entry['student_id'] == STUDENT_ID else ''
    print(f"  {rank:>2}. {entry['student_id']:<20} {entry['final_score']:.4f}{marker}")

## 6. Where to improve

If your score is below 0.85, the per-dimension breakdown in `judge_metadata` tells you which dimension to attack first. Common bottlenecks for this baseline and what to try:

| If your weak dimension is… | …consider |
|---|---|
| **correctness** | The model is mis-answering. Try sharper system prompt; raise `TOP_K`; switch to `claude-sonnet-4-6` for harder questions. |
| **faithfulness** | The model is making things up. Tighten the system prompt's "ONLY the chunks" line; add a self-check turn that asks the model to verify each claim cites a chunk. |
| **retrieval** | Your `retrieved_contexts` is missing relevant chunks. Augment BM25 with dense retrieval (e.g. Anthropic embeddings or a tiny sentence-transformer) and union the top-k. |
| **citations** | The cited IDs disagree with gold. Check `parse_citations` — does the model emit the right format? Bias the prompt with a worked example showing the bracket pattern. |
| **cost** | You're spending more than baseline. Drop to a smaller model on easy questions; cap `max_tokens`; cache repeated chunks via Anthropic's prompt caching. |
| **safety** | Your answer mentions something harmful or off-topic. Tighten the system prompt's refusal rule; reject answers that don't cite at least one chunk. |

Good luck.